In [ ]:
import pandas as pd
import networkx as nx
import itertools

excel_file_path = './data/Menstrual_cramps.xlsx'

node_df = pd.read_excel(excel_file_path, sheet_name='경혈단독_count')

invalid_nodes = ['non-acupoint', 'ashi-point', 'BV', 'CV', 'FV']
valid_node_df = node_df[~node_df['경혈명'].isin(invalid_nodes)].copy()
valid_node_df['경혈명'] = valid_node_df['경혈명'].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)
valid_nodes_set = set(valid_node_df['경혈명'])

print(f"유효 노드: {len(valid_nodes_set)}개\n")


df_meta = pd.read_excel(excel_file_path, sheet_name='논문_경혈정리')

# Year가 있는 논문만 필터링
df_meta_clean = df_meta.dropna(subset=['Year']).copy()
df_meta_clean['Year'] = df_meta_clean['Year'].astype(int)

print(f"   - Year 정보가 있는 논문: {len(df_meta_clean)}개")
print(f"   - 연도 범위: {df_meta_clean['Year'].min()} ~ {df_meta_clean['Year'].max()}")

# Era 할당 함수
def assign_era(y):
    if y <= 2010:
        return "Era1_1987_2010"
    elif y <= 2016:
        return "Era2_2011_2016"
    else:
        return "Era3_2017_2022"

df_meta_clean['Era'] = df_meta_clean['Year'].apply(assign_era)

print("\n   Era별 논문 수:")
for era in ["Era1_1987_2010", "Era2_2011_2016", "Era3_2017_2022"]:
    count = (df_meta_clean['Era'] == era).sum()
    print(f"   - {era}: {count}개")


all_edges_era = []

for _, row in df_meta_clean.iterrows():
    era = row['Era']
    acupoints_str = str(row['Acupoints'])

    # 경혈 분리 및 정제
    acupoints_list = [p.strip().replace('Kl', 'KI')
                      for p in acupoints_str.replace(',', ' ').split()]

    # 유효 경혈만 필터링
    valid_acu = [p for p in acupoints_list if p in valid_nodes_set]

    # 2개 이상의 경혈이 있을 때 조합으로 엣지 생성
    if len(valid_acu) >= 2:
        edges = list(itertools.combinations(sorted(valid_acu), 2))
        for source, target in edges:
            all_edges_era.append((source, target, era))

# DataFrame 생성
df_long_edges_era = pd.DataFrame(all_edges_era, columns=['Source', 'Target', 'Era'])

print(f"엣지 생성 완료: {len(df_long_edges_era)}개")

print("\n   Era별 엣지 분포:")
for era in ["Era1_1987_2010", "Era2_2011_2016", "Era3_2017_2022"]:
    count = (df_long_edges_era['Era'] == era).sum()
    print(f"   - {era}: {count}개")

print("\n[샘플 데이터]")
print(df_long_edges_era.head(10))

print("\n" + "="*60)
print("df_long_edges_era 생성 완료")

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

df_source = df_long_edges_era

table1_acupoints = [
    'SP6', 'CV4', 'ST36', 'CV6', 'SP8', 'BL32', 'CV3', 'LR3', 'EX-CA1', 'SP10', 'LI4', 'SP4', 'BL23', 'ST29', 'ST25'
]
era_order = ["Era1_1987_2010", "Era2_2011_2016", "Era3_2017_2022"]

data = []
for era in era_order:
    subset = df_source[df_source['Era'] == era]
    if subset.empty: continue

    G_era = nx.from_pandas_edgelist(subset, 'Source', 'Target')
    cent_scores = nx.degree_centrality(G_era)

    for p in table1_acupoints:
        score = cent_scores.get(p, 0)
        data.append({'Era': era, 'Acupoint': p, 'Centrality': score})

df_cent_plot = pd.DataFrame(data)

def optimize_label_positions(items, min_dist=0.03):
    """라벨들이 서로 겹치지 않도록 y 좌표를 조정"""
    if not items: return []

    # y값 기준 정렬
    items.sort(key=lambda x: x['y'])

    # 1. 서로 밀어내기 (아래에서 위로)
    for i in range(1, len(items)):
        prev = items[i-1]
        curr = items[i]
        if curr['y'] - prev['y'] < min_dist:
            curr['y'] = prev['y'] + min_dist

    # 2. 중심 보정 (원래 위치의 평균으로 되돌리기)
    original_mean = np.mean([item['original_y'] for item in items])
    new_mean = np.mean([item['y'] for item in items])
    shift = original_mean - new_mean
    for item in items:
        item['y'] += shift

    return items


if df_cent_plot.empty:
    print("출력할 데이터가 없습니다.")
else:
    plt.figure(figsize=(18, 12))

    x_coords = [0, 1, 2]
    x_labels = ['Era 1\n(1987-2010)', 'Era 2\n(2011-2016)', 'Era 3\n(2017-2022)']
    colors = plt.cm.tab20(np.linspace(0, 1, len(table1_acupoints)))

    # (1) 선 그래프 그리기
    for idx, p in enumerate(table1_acupoints):
        subset = df_cent_plot[df_cent_plot['Acupoint'] == p]
        if subset.empty: continue

        scores = []
        for era in era_order:
            val = subset[subset['Era'] == era]['Centrality'].values
            scores.append(val[0] if len(val) > 0 else 0)

        plt.plot(x_coords, scores, marker='o', markersize=6, linewidth=2.5,
                 color=colors[idx], linestyle='-', alpha=0.6, zorder=2)

    # (2) 라벨링 (Era 2 개선 로직 적용)
    for x_idx, era in enumerate(era_order):

        # 해당 Era의 데이터 수집
        label_items = []
        for idx, p in enumerate(table1_acupoints):
            subset = df_cent_plot[(df_cent_plot['Acupoint'] == p) & (df_cent_plot['Era'] == era)]
            score = subset['Centrality'].values[0] if not subset.empty else 0

            label_items.append({
                'original_y': score,
                'y': score,
                'text': f"{p} ({score:.3f})", # (이름, 점수)
                'color': colors[idx],
                'acupoint': p
            })

        # Era 2인 경우 좌우 분산 배치 ---
        if x_idx == 1: # Era 2 (중간)
            # 점수순 정렬
            label_items.sort(key=lambda x: x['original_y'])

            left_items = []
            right_items = []

            # 하나씩 번갈아가며 좌/우 리스트에 담기
            for i, item in enumerate(label_items):
                if i % 2 == 0:
                    left_items.append(item)
                else:
                    right_items.append(item)

            # 각각 최적화 수행 (공간 활용 극대화)
            left_items = optimize_label_positions(left_items, min_dist=0.04)
            right_items = optimize_label_positions(right_items, min_dist=0.04)

            # 그리기 함수 (내부 헬퍼)
            def draw_era2_labels(items, side):
                for item in items:
                    if side == 'left':
                        txt_x = x_coords[x_idx] - 0.15  # 왼쪽으로 더 뺌
                        ha = 'right'
                        link_x = x_coords[x_idx] - 0.02
                    else:
                        txt_x = x_coords[x_idx] + 0.15  # 오른쪽으로 더 뺌
                        ha = 'left'
                        link_x = x_coords[x_idx] + 0.02

                    plt.text(txt_x, item['y'], item['text'],
                             ha=ha, va='center', fontsize=10, color=item['color'], fontweight='bold',
                             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.5))

                    # 연결선
                    plt.plot([link_x, txt_x], [item['original_y'], item['y']],
                             color='gray', lw=0.8, alpha=0.5, ls=':')

            draw_era2_labels(left_items, 'left')
            draw_era2_labels(right_items, 'right')

        else:
            # --- Era 1, 3은 기존 방식 (한쪽 정렬) ---
            optimized_labels = optimize_label_positions(label_items, min_dist=0.035)

            for item in optimized_labels:
                if x_idx == 0: # Era 1 (왼쪽 정렬)
                    ha = 'right'
                    txt_x = x_coords[x_idx] - 0.05
                    link_x = x_coords[x_idx] - 0.01
                else:          # Era 3 (오른쪽 정렬)
                    ha = 'left'
                    txt_x = x_coords[x_idx] + 0.05
                    link_x = x_coords[x_idx] + 0.01

                plt.text(txt_x, item['y'], item['text'],
                         ha=ha, va='center', fontsize=10, color=item['color'], fontweight='bold',
                         bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.5))

                if abs(item['y'] - item['original_y']) > 0.005:
                    plt.plot([link_x, txt_x], [item['original_y'], item['y']],
                             color='gray', lw=0.8, alpha=0.5, ls=':')

    plt.xticks(x_coords, x_labels, fontsize=14, fontweight='bold')
    plt.yticks(fontsize=12)
    plt.ylabel("Degree Centrality Score", fontsize=16, fontweight='bold')
    plt.title("Era Degree Centrality Score", fontsize=22, fontweight='bold', pad=30)
    plt.grid(axis='y', linestyle=':', alpha=0.4)
    plt.margins(x=0.2)
    plt.tight_layout()

    plt.savefig("Acupoint_Trend_Zigzag.png", dpi=300, bbox_inches='tight')
    plt.show()